# claudelocal on Kaggle - Qwen3.8-27B + GLM-4.7-Flash

Serves both models from one Kaggle **T4 x2** node and exposes them to `claudelocal`
on your own machine through a Cloudflare tunnel.

| | |
|---|---|
| Qwen3.8-27B UD-Q4_K_M | 15.3 GB |
| GLM-4.7-Flash UD-Q4_K_XL | 16.3 GB |
| Both on disk | 31.7 GB - fits `/tmp` (~60 GB) |
| One resident at a time | ~22 GB of 32 GB VRAM, fully GPU-offloaded |

They cannot both be resident (31.7 GB of weights against 32 GB of VRAM), so
[llama-swap](https://github.com/mostlygeek/llama-swap) loads whichever one a request asks
for - the same one-at-a-time behaviour `claudelocal` already has locally.

## Before you run this

1. **Settings -> Accelerator -> GPU T4 x2**
2. **Settings -> Internet -> On**
3. *(recommended)* **Settings -> Persistence -> Files only** - keeps the compiled
   `llama-server` in `/kaggle/working`, so later sessions skip the ~15 minute CUDA build.

Then **Run All**. The last cell prints a `remotes.json` block to paste on your own machine.

> A Kaggle session is capped at **9 hours** and free-tier GPU quota is **30 h/week**.
> The tunnel URL is new every session, so `remotes.json` needs re-pasting each time.


## 1. Configuration


In [ ]:
import os, secrets

# /tmp has ~60 GB and is wiped between sessions; /kaggle/working has 20 GB and survives when
# Persistence is on. So the compiled binaries live there and the 32 GB of weights do not.
MODEL_DIR = '/tmp/models'
CACHE_DIR = '/kaggle/working/llamacpp'
SWAP_PORT = 8080

# sm_75 is the T4. Building one architecture instead of all of them is most of the reason
# this takes ~15 minutes rather than an hour.
CUDA_ARCH = '75'
LLAMA_TAG = 'b10850'

# Every model in claudelocal's picker shares one context window and the smallest sets it, so
# this has to be at least the window claudelocal reports locally (128000 by default).
CTX_SIZE = 131072

MODELS = {
    'qwen3.8-27b': {
        'repo': 'unsloth/Qwen3.8-27B-GGUF',
        'file': 'Qwen3.8-27B-UD-Q4_K_M.gguf',
        'name': 'Qwen3.8-27B',
    },
    'glm-4.7-flash': {
        'repo': 'unsloth/GLM-4.7-Flash-GGUF',
        'file': 'GLM-4.7-Flash-UD-Q4_K_XL.gguf',
        'name': 'GLM-4.7-Flash',
    },
}

# A Cloudflare quick tunnel is a public URL. llama-swap checks this key on every request, so
# the endpoint is not open to anyone who guesses the hostname.
API_KEY = 'sk-cl-' + secrets.token_urlsafe(32)

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)
print('API key for this session:', API_KEY)


## 2. GPU and disk check


In [ ]:
import subprocess

!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
!echo && df -h /tmp /kaggle/working

gpus = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.strip()
if not gpus:
    raise SystemExit('No GPU. Settings -> Accelerator -> GPU T4 x2, then Run All again.')
print()
print(gpus)
if len(gpus.splitlines()) < 2:
    print()
    print('WARNING: only one GPU visible. Expected T4 x2 for 32 GB total.')
    print('         A single 16 GB T4 is under the ~22 GB these models need.')


## 3. Build llama.cpp with CUDA

llama.cpp publishes no prebuilt CUDA binary for Linux, so it is compiled here - for sm_75
only, and only the `llama-server` target. With Persistence on this runs once.


In [ ]:
import os, subprocess, shutil

server_bin = os.path.join(CACHE_DIR, 'bin', 'llama-server')

if os.path.exists(server_bin):
    print('Using the cached build at', server_bin)
else:
    print('Building llama.cpp ' + LLAMA_TAG + ' for sm_' + CUDA_ARCH + ' - about 15 minutes.')
    subprocess.run('apt-get -qq update && apt-get -qq install -y ninja-build',
                   shell=True, check=False)
    if not os.path.isdir('/tmp/llama.cpp'):
        subprocess.run('git clone --depth 1 -b ' + LLAMA_TAG +
                       ' https://github.com/ggml-org/llama.cpp /tmp/llama.cpp',
                       shell=True, check=True)
    subprocess.run('cmake -S /tmp/llama.cpp -B /tmp/llama.cpp/build -G Ninja '
                   '-DCMAKE_BUILD_TYPE=Release -DGGML_CUDA=ON '
                   '-DCMAKE_CUDA_ARCHITECTURES=' + CUDA_ARCH + ' '
                   '-DGGML_NATIVE=OFF -DLLAMA_CURL=OFF '
                   '-DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_EXAMPLES=OFF',
                   shell=True, check=True)
    subprocess.run('cmake --build /tmp/llama.cpp/build --target llama-server -j$(nproc)',
                   shell=True, check=True)
    # The build emits shared libraries beside the binary; both have to be kept.
    os.makedirs(os.path.join(CACHE_DIR, 'bin'), exist_ok=True)
    built = '/tmp/llama.cpp/build/bin'
    for f in os.listdir(built):
        src = os.path.join(built, f)
        if os.path.isfile(src) and (f == 'llama-server' or '.so' in f):
            shutil.copy2(src, os.path.join(CACHE_DIR, 'bin', f))
    print('Cached the build in', CACHE_DIR)

os.environ['LD_LIBRARY_PATH'] = (os.path.join(CACHE_DIR, 'bin') + ':' +
                                 os.environ.get('LD_LIBRARY_PATH', ''))
subprocess.run([server_bin, '--version'], check=False)


## 4. Install llama-swap and cloudflared


In [ ]:
import os, json, tarfile, urllib.request, subprocess

def latest_asset(repo, match):
    url = 'https://api.github.com/repos/' + repo + '/releases/latest'
    with urllib.request.urlopen(url) as r:
        rel = json.load(r)
    for a in rel['assets']:
        if match(a['name']):
            return rel['tag_name'], a['browser_download_url']
    raise SystemExit('no matching asset in ' + repo)

if not os.path.exists('/usr/local/bin/llama-swap'):
    tag, url = latest_asset('mostlygeek/llama-swap',
                            lambda n: 'linux' in n and 'amd64' in n and n.endswith('.tar.gz'))
    print('llama-swap', tag)
    urllib.request.urlretrieve(url, '/tmp/llama-swap.tar.gz')
    with tarfile.open('/tmp/llama-swap.tar.gz') as t:
        t.extractall('/tmp/llama-swap-dist')
    for root, _, files in os.walk('/tmp/llama-swap-dist'):
        if 'llama-swap' in files:
            shutil_src = os.path.join(root, 'llama-swap')
            os.replace(shutil_src, '/usr/local/bin/llama-swap')
            break
    os.chmod('/usr/local/bin/llama-swap', 0o755)

if not os.path.exists('/usr/local/bin/cloudflared'):
    urllib.request.urlretrieve(
        'https://github.com/cloudflare/cloudflared/releases/latest/download/'
        'cloudflared-linux-amd64', '/usr/local/bin/cloudflared')
    os.chmod('/usr/local/bin/cloudflared', 0o755)

subprocess.run(['/usr/local/bin/llama-swap', '--version'], check=False)
subprocess.run(['/usr/local/bin/cloudflared', '--version'], check=False)


## 5. Download the weights

About 32 GB into `/tmp`. `hf_transfer` saturates Kaggle's link, so this is usually well
under ten minutes. `/tmp` is wiped between sessions, so it runs every time.


In [ ]:
!pip install -q hf_transfer huggingface_hub

import os
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
from huggingface_hub import hf_hub_download

paths = {}
for key, m in MODELS.items():
    print('-> ' + m['file'])
    paths[key] = hf_hub_download(repo_id=m['repo'], filename=m['file'], local_dir=MODEL_DIR)

print()
for k, p in paths.items():
    print('%-16s %6.2f GB  %s' % (k, os.path.getsize(p) / 1024 ** 3, p))


## 6. Write the llama-swap config


In [ ]:
import os, json

bin_dir = os.path.join(CACHE_DIR, 'bin')

# One shared macro carries the flags both models want, so each model entry is just its weights.
# -fa auto enables flash attention where the T4 supports it, and q8_0 KV keeps a 131072-token
# window inside VRAM alongside a ~16 GB model.
#
# json.dumps is used wherever a YAML scalar needs quoting: JSON strings are valid YAML, which
# saves hand-escaping the key and the model ids.
lines = [
    'healthCheckTimeout: 1800',
    'logLevel: info',
    'logToStdout: both',
    '',
    'apiKeys:',
    '  - ' + json.dumps(API_KEY),
    '',
    'macros:',
    '  server: >',
    '    ' + bin_dir + '/llama-server',
    '    --host 127.0.0.1 --port ${PORT}',
    '    --jinja -fa auto',
    '    --n-gpu-layers 999',
    '    --ctx-size ' + str(CTX_SIZE),
    '    --cache-type-k q8_0 --cache-type-v q8_0',
    '',
    'models:',
]

for key, m in MODELS.items():
    lines += [
        '  ' + json.dumps(key) + ':',
        '    name: ' + json.dumps(m['name']),
        '    cmd: |',
        '      ${server}',
        '      --model ' + paths[key],
        '      --alias ' + key,
        '',
    ]

cfg = chr(10).join(lines)
CONFIG_PATH = '/kaggle/working/llama-swap.yaml'
with open(CONFIG_PATH, 'w') as f:
    f.write(cfg)
print(cfg)


## 7. Start the server and open the tunnel


In [ ]:
import os, re, time, threading, subprocess, urllib.request

env = dict(os.environ)
env['LD_LIBRARY_PATH'] = os.path.join(CACHE_DIR, 'bin')

swap_log = open('/kaggle/working/llama-swap.log', 'w')
swap = subprocess.Popen(
    ['/usr/local/bin/llama-swap', '--config', CONFIG_PATH,
     '--listen', '127.0.0.1:' + str(SWAP_PORT)],
    stdout=swap_log, stderr=subprocess.STDOUT, env=env)

# llama-swap answers /health immediately - it loads no model until a request names one.
up = False
for _ in range(60):
    try:
        urllib.request.urlopen('http://127.0.0.1:%d/health' % SWAP_PORT, timeout=2).read()
        up = True
        break
    except Exception:
        time.sleep(1)
if not up:
    raise SystemExit('llama-swap did not start - see /kaggle/working/llama-swap.log')
print('llama-swap is up on', SWAP_PORT)

# A quick tunnel needs no Cloudflare account. It prints its hostname once and then goes quiet,
# so the URL is scraped from the first seconds of output.
tunnel = subprocess.Popen(
    ['/usr/local/bin/cloudflared', 'tunnel', '--no-autoupdate',
     '--url', 'http://127.0.0.1:' + str(SWAP_PORT)],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

PUBLIC_URL = None
deadline = time.time() + 90
while time.time() < deadline:
    line = tunnel.stdout.readline()
    if not line:
        break
    hit = re.search(r'https://[a-z0-9][a-z0-9-]*\.trycloudflare\.com', line)
    if hit:
        PUBLIC_URL = hit.group(0)
        break

if not PUBLIC_URL:
    raise SystemExit('cloudflared reported no URL - re-run this cell.')

# Keep draining the pipe, or cloudflared blocks on a full buffer and the tunnel stalls.
threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()
print('tunnel:', PUBLIC_URL)


## 8. Verify, then copy this into `remotes.json`


In [ ]:
import json, time, urllib.request

# The first request to a model loads ~16 GB from /tmp, so it is slow; later ones are not.
def probe(model):
    body = json.dumps({
        'model': model, 'max_tokens': 16,
        'messages': [{'role': 'user', 'content': 'Reply with the single word: ready'}],
    }).encode()
    req = urllib.request.Request(PUBLIC_URL + '/v1/messages', data=body, method='POST',
                                 headers={
                                     'content-type': 'application/json',
                                     'anthropic-version': '2023-06-01',
                                     'authorization': 'Bearer ' + API_KEY,
                                 })
    t0 = time.time()
    with urllib.request.urlopen(req, timeout=1800) as r:
        out = json.load(r)
    text = ''.join(b.get('text', '') for b in out.get('content', []))
    return time.time() - t0, text.strip()

for key in MODELS:
    try:
        secs, text = probe(key)
        print('ok    %-16s %6.1fs  %r' % (key, secs, text))
    except Exception as e:
        print('FAIL  %-16s %s' % (key, e))

block = {}
for k, m in MODELS.items():
    block[k] = {
        'label': m['name'] + ' (Kaggle T4x2)',
        'baseUrl': PUBLIC_URL,
        'model': k,
        'apiKey': API_KEY,
        'ctx': CTX_SIZE,
        'note': 'Kaggle session - this URL changes every time the notebook restarts',
    }

print()
print('=' * 78)
print('Paste into remotes.json in your claudelocal folder, then run:')
print('    claudelocal remotes --check')
print('=' * 78)
print(json.dumps(block, indent=2))


## 9. Keep the session alive

This cell runs until you stop it or Kaggle's 9-hour cap ends the session. Leave it running
for as long as you want the endpoint reachable.


In [ ]:
import time, datetime, urllib.request

started = time.time()
try:
    while True:
        try:
            urllib.request.urlopen('http://127.0.0.1:%d/health' % SWAP_PORT, timeout=5).read()
            state = 'ok'
        except Exception as e:
            state = 'unhealthy: %s' % e
        print('[%s] %s  up %.2fh / 9h  %s' % (
            datetime.datetime.now().strftime('%H:%M:%S'), state,
            (time.time() - started) / 3600, PUBLIC_URL), flush=True)
        time.sleep(300)
except KeyboardInterrupt:
    print('stopped')
